# Extract satellite patches from the Mesogeos zarr cube

Cuts a `WIN x WIN` window (4 channels: NDVI, LAI, LST day, LST night) around each
Track A sample's cell on its last observed day (t-1), and saves npz shards.
These are the inputs for the ViT branch (Models A, C, D).

**Before running (one-time):** open the shared
[mesogeos Drive folder](https://drive.google.com/drive/folders/1aRXQXVvw6hz0eYgtJDoixjPQO-_bRKz9),
right-click `mesogeos_cube.zarr` → *Organise* → *Add shortcut* → My Drive.
Then Runtime → Run all. Safe to re-run: already-written shards are skipped.

In [ ]:
!pip -q install zarr xarray
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, pandas as pd, xarray as xr, time
from pathlib import Path

WIN = 64                      # window size in km/cells (64 -> 16 ViT patches of 16x16)
SHARD = 2000                  # samples per output shard
VARS = ['ndvi', 'lai', 'lst_day', 'lst_night']
CUBE = '/content/drive/MyDrive/mesogeos_cube.zarr'
OUT = Path('/content/drive/MyDrive/wildfire_patches')   # shards land in your Drive
OUT.mkdir(exist_ok=True)

MANIFEST = 'https://raw.githubusercontent.com/Aaffnnaann/wildfire-dissertation/main/manifest.csv'
mf = pd.read_csv(MANIFEST, parse_dates=['date'])
print(len(mf), 'samples')

ds = xr.open_zarr(CUBE, consolidated=True)
xs, ys = ds.x.values, ds.y.values          # cube coordinate axes
print({v: ds[v].shape for v in VARS})

In [ ]:
H = WIN // 2

def window(lon, lat, date):
    ix = int(np.abs(xs - lon).argmin())
    iy = int(np.abs(ys - lat).argmin())
    x0, x1 = ix - H, ix + H
    y0, y1 = iy - H, iy + H
    sel = ds[VARS].sel(time=date, method='nearest').isel(
        x=slice(max(x0, 0), min(x1, len(xs))),
        y=slice(max(y0, 0), min(y1, len(ys))))
    arr = np.stack([sel[v].values for v in VARS], -1).astype(np.float32)
    # pad to WIN x WIN if the window was clipped at the cube border
    out = np.full((WIN, WIN, len(VARS)), np.nan, np.float32)
    oy, ox = max(0, -y0), max(0, -x0)
    out[oy:oy + arr.shape[0], ox:ox + arr.shape[1]] = arr
    return out

# timing check on 5 samples before committing to the full run
t0 = time.time()
for _, r in mf.head(5).iterrows():
    window(r.lon, r.lat, r.date)
per = (time.time() - t0) / 5
print(f'{per:.2f}s/sample -> full run ~{per * len(mf) / 3600:.1f}h')

In [ ]:
for split, g in mf.groupby('split'):
    g = g.sort_values('idx')
    for s0 in range(0, len(g), SHARD):
        chunk = g.iloc[s0:s0 + SHARD]
        path = OUT / f'{split}_{s0:05d}.npz'
        if path.exists():
            continue
        t0 = time.time()
        patches = np.stack([window(r.lon, r.lat, r.date) for _, r in chunk.iterrows()])
        np.savez_compressed(path, patches=patches, idx=chunk['idx'].to_numpy())
        print(f'{path.name}: {patches.shape} nan%={np.isnan(patches).mean():.3f} '
              f'({time.time() - t0:.0f}s)')
print('done')

In [ ]:
# sanity check: visualise one fire sample's 4 channels
import matplotlib.pyplot as plt
d = np.load(sorted(OUT.glob('train_*.npz'))[0])
i = int(np.argmax(mf[mf.split == 'train'].sort_values('idx')['y'].to_numpy()[d['idx']]))
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for k, (ax, v) in enumerate(zip(axes, VARS)):
    ax.imshow(d['patches'][i, :, :, k])
    ax.set_title(v)
    ax.axis('off')
plt.show()